In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np
import math
from scipy.stats import norm
import matplotlib.pyplot as plt
import sys
from openpyxl import Workbook

In [2]:
#Entry
tperiod = 8

# Specify the instance version here

In [4]:
# Specify the instance version
instance_version = "largesize" #Versions: "original", "smallsize", "mediumsize", "largesize"

# Define file names dynamically based on instance version
location_file = f"fullDist Set V Data Location-{instance_version}.xlsx"
county_file = f"fullDist Set C Data-{instance_version}.xlsx"
resource_file = f"fullDist R-{instance_version}.xlsx"
time_matrix_file = f"durationFull-{instance_version}.txt"
distance_matrix_file = f"distanceFull-{instance_version}.txt"
inputs_file = f"fullDistInputs-{instance_version}.xlsx"
k_ijt_file = f"K_ijt_data_fixed-{instance_version}.xlsx"

# Specify the number of scenarios here

In [6]:
# Define the number of scenarios
num_sce = 5 # Example: 10 scenarios (adjust as needed)
sce = list(range(num_sce)) 

# Generate equal probabilities for each scenario
P = [1/num_sce] * num_sce

#Demand data
demand_file = f"demand_data_{instance_version}.xlsx"

# Dynamically generate sheet names based on the number of scenarios
sheet_names = [f"Demand_{i+1}" for i in sce]

print(f"Number of scenarios: {num_sce}")
print(f"Generated sheet names: {sheet_names}")

Number of scenarios: 5
Generated sheet names: ['Demand_1', 'Demand_2', 'Demand_3', 'Demand_4', 'Demand_5']


In [7]:
# Location Data
df_V = pd.read_excel(location_file)
print(df_V.head())

# County Data
df_C = pd.read_excel(county_file)
print(df_C.head())

# Resource Data
df_R = pd.read_excel(resource_file)
print(df_R.head())

# Time and Distance Matrices
timeMatrix = np.loadtxt(time_matrix_file)  # Time duration matrix (seconds)
distanceMatrix = np.loadtxt(distance_matrix_file)  # Distance matrix
print(timeMatrix)
print(distanceMatrix)

# Inputs Data
df_inputs = pd.read_excel(inputs_file)
print(df_inputs.head())

# K_ijt Data
df_K = pd.read_excel(k_ijt_file)
print(df_K.head())

   Index County       Type  Cluster                         Facility  \
0      0    NaN  Warehouse      NaN        SCEMD Winnsboro Warehouse   
1      1    NaN  Warehouse      4.0   US Military Base - Cot Storage   
2      2    NaN   Supplier      NaN                   SOPAKCO (Food)   
3      3    NaN   Supplier      NaN                   Lipsey (Water)   
4      4    NaN   Supplier      NaN  FEMA Warehouse (Food and Water)   

                                             Address   latitude  longitude  \
0              100 Tidewater Dr, Winnsboro, SC 29180 -81.098402  34.403608   
1                Garners Ferry Rd, Hopkins, SC 29061 -80.833969  34.045263   
2                118 S Cypress St, Mullins, SC 29574 -79.262558  34.202395   
3               1701 Oakbrook Dr, Norcross, GA 30093 -84.196174  33.913660   
4  3780 Southside Industrial Pkwy, Atlanta, GA 30354 -84.373590  33.650208   

   Shelter Capacity  Pallet Capacity  ...  P-Kit StartInv  Water StartInv  \
0               NaN  

In [8]:
# Load demand data
d_its = {} # Dictionary to store demand data in (location, time, scenario) format

for s, sheet in enumerate(sheet_names):  # Scenario index starts from 0
    df = pd.read_excel(demand_file, sheet_name=sheet, header=0, index_col=0) # Read demand data for the current sheet
    
    # Replace NaN values with 0 (if applicable)
    df = df.fillna(0)
    # Convert DataFrame to a NumPy array for iteration
    demand_array = df.values
    
    # Process demand data for each location and time period
    for location_index in range(demand_array.shape[0]): #Rows represent locations
        for time_period in range(demand_array.shape[1]): #Columns represent time periods
            d_its[(location_index, time_period, s)] = demand_array[location_index, time_period]

# Print a sample to verify the correct location
print("Sample demand data for location 5, time 2, scenario 0:", d_its[(5, 2, 0)])
print("Sample demand data for location 6, time 2, scenario 0:", d_its[(6, 2, 0)])

Sample demand data for location 5, time 2, scenario 0: 0.0
Sample demand data for location 6, time 2, scenario 0: 105.6570290811453


In [9]:
# Defining sets

# County Set
C = len(df_C)
print('All counties:', C)

# Location Sets
V = len(df_V)
print('All locations:', V)

# Subsets of Location Set (V0, V1, V2, V3)
V0 = []  # Warehouses and Suppliers
V1 = []  # Shelters
V2 = []  # RSAs (Staging Areas)
V3 = []  # PoDs (Points of Distribution)


for i in range(V):
    if df_V['Type'][i] == 'Warehouse' or df_V['Type'][i] == 'Supplier':
        V0.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'Shelter':
        V1.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'RSA':
        V2.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'PoD':
        V3.append(df_V['Index'][i])

# Print lengths of subsets
print('Warehouse/Supplier locations:', len(V0), V0)
print('Shelter locations:', len(V1), V1)
print('RSA locations:', len(V2), V2)
print('PoD locations:', len(V3), V3)

# Commodity Set
R = len(df_R)
print('All resources:', R)

# Time Period Set
T = int(df_inputs['Hours in 6 days'][0] / tperiod)
print('Time periods:', T)

#Time Period Subsets
T_Bef = list(range(int(T / 2)))
T_Aft = list(range(int(T / 2), T))
print('Time Before Landfall:', T_Bef)
print('Time After Landfall:', T_Aft)

# Print to verify
print("Scenario indices:", sce)
print("Scenario probabilities:", P)

All counties: 6
All locations: 24
Warehouse/Supplier locations: 6 [0, 1, 2, 3, 4, 5]
Shelter locations: 6 [6, 7, 8, 9, 10, 11]
RSA locations: 6 [12, 13, 14, 15, 16, 17]
PoD locations: 6 [18, 19, 20, 21, 22, 23]
All resources: 6
Time periods: 10
Time Before Landfall: [0, 1, 2, 3, 4]
Time After Landfall: [5, 6, 7, 8, 9]
Scenario indices: [0, 1, 2, 3, 4]
Scenario probabilities: [0.2, 0.2, 0.2, 0.2, 0.2]


In [10]:
#Create Arc Set where cannot go from shelter to PoD

A = set() #set to store arc pairs

no_demand=[]
for i in set(V1).union(set(V3)):
    i_has_demand = any(df_V['County'][i] == df_C['County'][c] and df_C['Shelter Total Demand'][c] > 0 for c in range(C))
    if not i_has_demand:
        no_demand.append(i)
print(no_demand)

for i in range(V):
    for j in range(V):
        if i != j and i not in no_demand and j not in no_demand:
            if (i not in V1 or j not in V3) and (i not in V3 or j not in V1):
                A.add((i, j))

#print("Arc set:", arc_set)
print("Number of arcs:", len(A))
print("Number of arcs eliminated by this arc set:", (V*V)-len(A))
#print(A)

[]
Number of arcs: 480
Number of arcs eliminated by this arc set: 96


In [11]:
#Defining parameters

# Initialize inventory list
Inv = []

# Define starting inventory based on instance version
if instance_version == "original":
    for i in range(V):
        Inv.extend([
            df_V['S-Kit StartInv'][i],
            df_V['P-Kit StartInv'][i],
            df_V['Water StartInv'][i],
            df_V['Food StartInv'][i],
            df_V['Cot StartInv'][i],
            df_V['Blanket StartInv'][i]
        ])
    I_ir0 = [Inv[i:i+R] for i in range(0, len(Inv), R)]
    
elif instance_version == "largesize":
    for i in range(V):
        Inv.extend([
            df_V['S-Kit StartInv'][i],
            df_V['P-Kit StartInv'][i],
            df_V['Water StartInv'][i],
            df_V['Food StartInv'][i],
            df_V['Cot StartInv'][i],
            df_V['Blanket StartInv'][i]
        ])
    I_ir0 = [Inv[i:i+R] for i in range(0, len(Inv), R)]

elif instance_version == "smallsize":
    for i in range(V):
        Inv.append(df_V['Water StartInv'][i])
    I_ir0 = [[inv] for inv in Inv]

elif instance_version == "mediumsize":
    for i in range(V):
        Inv.extend([
            df_V['Water StartInv'][i],
            df_V['Blanket StartInv'][i]
        ])
    I_ir0 = [Inv[i:i+R] for i in range(0, len(Inv), R)]


#penalty cost for location i p_i[I]
p_i = df_V['P_i'].tolist()  # Convert the column directly to a list

#time periods to travel i,j t_ijt[i][j]
seconds = 3600
t_ijt = [[math.ceil((timeMatrix[i][j] / seconds) / tperiod) for j in range(V)] for i in range(V)]

#Fraction of a truck 1 pallet of resource r requires w_r[R]
w_r = [1 / pallet for pallet in df_R['Pallet Units on Truck']]
    
#Factor of pallet required per commodity r of kit g_shelter[R]
g_shelter = df_R['pallet per S-Kit'].tolist()

#Factor of pallet required per commodity r of kit g_pod[R]
g_pod = df_R['pallet per P-Kit'].tolist()

# Initialize K_ijt as a nested dictionary {i: {j: {t: capacity}}}
K_ijt = {}

# Populate K_ijt with data from df_K
for index, row in df_K.iterrows():
    i, j, t, capacity = row['From_Node'], row['To_Node'], row['Time_Period'], row['Capacity']
    K_ijt.setdefault(i, {}).setdefault(j, {})[t] = capacity

print(w_r)

[0.022727272727272728, 0.029411764705882353, 0.038461538461538464, 0.020833333333333332, 0.019230769230769232, 0.019230769230769232]


In [12]:
# Define the model
m = gp.Model("Two-Stage Stochastic Programming Model")

# # Set the optimization direction
# m.modelSense = GRB.MINIMIZE

Set parameter Username
Academic license - for non-commercial use only - expires 2025-07-29


In [13]:
# Define decision variables 
# i, j is position, r is commodity, t is time, and s is scenario

# Define first-stage decision variables
# I: Inventory of commodity r at node i at the start of time period t
I = {}

for i in range(V):
    for r in range(R):
        for t in range(T):
            I[i, r, t] = m.addVar(vtype=GRB.CONTINUOUS, name="I_%s_%s_%s" % (i, r, t))

# Define second-stage decision variables
U = {}  # Demand shortage at location i at time period t in scenario s

for i in range(V):
    for t in range(T):
        for s in sce:  # Iterate directly over the scenario set (e.g., 0 and 1)
            U[i, t, s] = m.addVar(vtype=GRB.CONTINUOUS, name="U_%s_%s_%s" % (i, t, s))

S = {}  # Demand satisfied at location i at time period t in scenario s

for i in range(V):
    for t in range(T):
        for s in sce:  # Iterate directly over the scenario set
            S[i, t, s] = m.addVar(vtype=GRB.CONTINUOUS, name="S_%s_%s_%s" % (i, t, s))

f = {}  # Quantity of commodity r traveling on link (i, j) at time period t in scenario s

for i in range(V):
    for j in range(V):
        if (i, j) in A:  # Only define f_ijrt^s if (i, j) is a valid link
            for r in range(R):
                for t in range(T):
                    for s in sce:  # Iterate directly over the scenario set
                        f[i, j, r, t, s] = m.addVar(vtype=GRB.CONTINUOUS, name="f_%s_%s_%s_%s_%s" % (i, j, r, t, s))
m.update()

In [14]:
# Define constraints

# Add constraints for the lower bound of inventory at warehouses (V0)
for i in V0:  # Loop over warehouse nodes
    for r in range(R):  # Loop over commodities
        for t in range(1, T):  # Loop over time periods (from t=1 to T-1)
            # Calculate the total arc capacity for the current time period
            arc_capacity = sum(K_ijt[i][j][t] for j in range(V) if (i, j) in A)
            # Calculate max outflow for commodity r
            max_outflow_r = arc_capacity  / w_r[r]
            # Check if the initial inventory is greater than or equal to max_outflow_r
            if I_ir0[i][r] >= max_outflow_r:
                # Add the lower bound constraint
                m.addConstr(I[i, r, t] >= I_ir0[i][r] - max_outflow_r, name=f"LowerBound_Warehouse_{i}_{r}_{t}")


# Add constraints for the upper bound of inventory at PoDs (V3)
for i in V3:  # PoDs
    for r in range(R):  # Commodities
        for t in T_Bef[1:]:  # Time periods before landfall, excluding t=0
            # Calculate the total arc capacity for the current time period
            arc_capacity = sum(K_ijt[j][i][t] for j in range(V) if (j, i) in A)
            # Calculate max inflow for commodity r
            max_inflow_r = arc_capacity / w_r[r]            
            # Add the upper bound constraint
            m.addConstr(I[i, r, t] <= I_ir0[i][r] + max_inflow_r, name=f"UpperBound_PoD_{i}_{r}_{t}")




# Add constraints to ensure inventory at warehouses only decreases over time
for i in V0:
    for r in range(R):
        for t in range(1, T):
            m.addConstr(
                I[i, r, t] <= I[i, r, t-1],
                name=f"inventory_decrease_warehouse_{i}_{r}_{t}")

# Add constraints to ensure inventory at PoDs only increases before landfall
for i in V3:
    for r in range(R):
        for t in T_Bef[1:]:
            m.addConstr(
                I[i, r, t] >= I[i, r, t-1],
                name=f"inventory_increase_pod_{i}_{r}_{t}")

m.update()  # Ensure the model is updated with the latest constraints
#m.write("model_constraints.lp")

#Starting Value of I[t=0]
for i in range(V):
    for r in range(R):
        m.addConstr(I[i,r,0]==I_ir0[i][r])

# Update demand shortage for all locations, time periods (except t=0), and scenarios
for i in range(V):
    for t in range(1, T):
        for s in sce:
            m.addConstr(U[i, t, s] == U[i, t-1, s] - S[i, t, s] + d_its[(i, t, s)], 
                        name="DemandShortageUpdate_%s_%s_%s" % (i, t, s))

# Initial demand shortage for all locations and scenarios at time period t=0
for i in range(V):
    for s in sce:
        m.addConstr(U[i, 0, s] == d_its[(i, 0, s)], name="InitialDemandShortage_%s_%s" % (i, s))

"""
# Demand satisfaction constraint for all locations, time periods, and scenarios
for i in range(V):
    for t in range(T):
        for s in sce:
            m.addConstr(S[i, t, s] <= U[i, t, s], name="DemandSatisfaction_%s_%s_%s" % (i, t, s))
"""
# Inventory update constraint for all locations, commodities, time periods (except last), and scenarios
for i in range(V):
    for r in range(R):
        for t in range(T - 1):
            for s in sce:
                outgoing_flow = gp.quicksum(f[i, j, r, t, s] for j in range(V) if (i, j) in A)
                incoming_flow = gp.quicksum(f[j, i, r, t - t_ijt[j][i] + 1, s] for j in range(V) if (j, i) in A and t - t_ijt[j][i] + 1 >= 0)
                
                # Inventory update constraint for shelters
                if i in V1:
                    m.addConstr(
                        I[i, r, t + 1] == I[i, r, t] - outgoing_flow + incoming_flow - S[i, t, s] * g_shelter[r],
                        name="InventoryUpdate_Shelter_%s_%s_%s_%s" % (i, r, t, s)
                    )
                # Inventory update constraint for pods
                elif i in V3:
                    m.addConstr(
                        I[i, r, t + 1] == I[i, r, t] - outgoing_flow + incoming_flow - S[i, t, s] * g_pod[r],
                        name="InventoryUpdate_Pod_%s_%s_%s_%s" % (i, r, t, s)
                    )
                # Inventory update constraint for warehouses and RSAs
                else:
                    m.addConstr(
                        I[i, r, t + 1] == I[i, r, t] - outgoing_flow + incoming_flow,
                        name="InventoryUpdate_%s_%s_%s_%s" % (i, r, t, s)
                    )
"""
# Inventory sufficiency constraint
for i in range(V):
    for r in range(R):
        for t in range(T):
            for s in sce:
                outgoing_flow = gp.quicksum(f[i, j, r, t, s] for j in range(V) if (i, j) in A)
                if i in V1:  # Shelter
                    m.addConstr(
                        I[i, r, t] >= outgoing_flow + S[i, t, s] * g_shelter[r],
                        name="InventorySufficiency_Shelter_%s_%s_%s_%s" % (i, r, t, s)
                    )
                elif i in V3:  # Pod
                    m.addConstr(
                        I[i, r, t] >= outgoing_flow + S[i, t, s] * g_pod[r],
                        name="InventorySufficiency_Pod_%s_%s_%s_%s" % (i, r, t, s)
                    )
                else:  # Warehouse or RSA
                    m.addConstr(
                        I[i, r, t] >= outgoing_flow,
                        name="InventorySufficiency_%s_%s_%s_%s" % (i, r, t, s)
                    )
"""

# Flow capacity constraint for all links, time periods, and scenarios
for i in range(V):
    for j in range(V):
        if (i, j) in A:  # Only consider valid links in A
            for t in range(T):
                for s in sce:
                    # Check if the capacity for this (i, j, t) exists in K_ijt
                    #if i in K_ijt and j in K_ijt[i] and t in K_ijt[i][j]:
                        m.addConstr(
                            gp.quicksum(w_r[r] * f[i, j, r, t, s] for r in range(R)) <= K_ijt[i][j][t],
                            name="FlowCapacity_%s_%s_%s_%s" % (i, j, t, s)
                        )
m.update()

In [15]:
# Set up the objective function
objective = gp.quicksum(
    P[s] * gp.quicksum(
        gp.quicksum(
            p_i[i] * U[i, t, s]
            for i in range(V)
        ) for t in range(T)
    )
    for s in sce  # Iterate over the defined scenario indices
)
m.update()

m.setObjective(objective, GRB.MINIMIZE)
m.update()

In [16]:
# Optimize the model
m.optimize()

# Check if the optimization was successful
if m.status == GRB.OPTIMAL:
    # Create an empty list to store variable values
    variable_data = []

    # Loop through decision variables and get their values
    for var in m.getVars():
        variable_data.append([var.VarName, var.X])  # Append variable name and value

    # Create a DataFrame from the collected data
    df = pd.DataFrame(variable_data, columns=['Variable', 'Value'])

    # Save the DataFrame to an Excel file
    output_file = 'decision_variables_output_small.xlsx'
    df.to_excel(output_file, index=False, engine='openpyxl')

    print(f'Decision variable values have been saved to {output_file}')
else:
    print("The model did not find an optimal solution.")

Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11+.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13800H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 32436 rows, 147840 columns and 422364 nonzeros
Model fingerprint: 0x42dcb32e
Coefficient statistics:
  Matrix range     [9e-03, 1e+00]
  Objective range  [2e-02, 1e-01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 6e+04]
Presolve removed 4172 rows and 16606 columns
Presolve time: 0.27s
Presolved: 28264 rows, 131234 columns, 393969 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 0.07s

Barrier performed 0 iterations in 0.45 seconds (0.31 work units)
Barrier solve interrupted - model solved by another algorithm


Solved with dual simplex
Iteration    Objective       Primal Inf.    Dual Inf.      Time
    1605    6.4219674e+03   0.000000e+